# AutoEconSentiment Transformer Walkthrough

This notebook shows how to use the optional transformer configuration path added to `auto-econ-sentiment`.

The first sections use the same included FOMC dataset as `autoecon_demo.ipynb`, so the examples run on actual repository data without downloading a Hugging Face model. They demonstrate:

- the original `Econ_Text_Algos`-style model list,
- conversion from `label_mapping` and `sentiment_values` to the package's internal `label_map`,
- sentence-level aggregation by `id_text`,
- harmonized positive/neutral/negative counts, shares, and net sentiment.

The final section shows the optional real-model workflow for environments installed with `uv sync --extra transformers`.

In [18]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

REPO_ROOT

PosixPath('/Users/cory/Desktop/auto-econ-sentiment')

In [19]:
import pandas as pd

from auto_econ_sentiment.pipeline import AutoEconSentiment
from auto_econ_sentiment.models.sentiment_transformers import SentimentTransformers

## 1. Original-style transformer config

The transformer config can use the older `Econ_Text_Algos` model-list style. Each model maps raw model labels to semantic labels with `label_mapping`, then maps semantic labels to numeric sentiment direction with `sentiment_values`.

In [20]:
transformer_config = {
    "enabled": True,
    "text_column_transformer": "text_clean",
    "aggregation_methods": ["sentence_pos"],
    "output_schema": "shares",
    "net_sentiment_formula": "positive_minus_negative",
    "models": [
        {
            "name": "gtfintechlab/FOMC-RoBERTa",
            "short_name": "fomc",
            "num_labels": 3,
            "max_length": 512,
            "batch_size": 8,
            "label_mapping": {
                "LABEL_0": "positive",
                "LABEL_1": "negative",
                "LABEL_2": "neutral",
            },
            "sentiment_values": {
                "positive": 1,
                "negative": -1,
                "neutral": 0,
            },
        },
        {
            "name": "ProsusAI/finbert",
            "short_name": "finbertpro",
            "num_labels": 3,
            "max_length": 512,
            "batch_size": 8,
            "label_mapping": {
                "neutral": "neutral",
                "positive": "positive",
                "negative": "negative",
            },
            "sentiment_values": {
                "neutral": 0,
                "positive": 1,
                "negative": -1,
            },
        },
    ],
}

expanded = AutoEconSentiment._expand_transformer_model_configs(
    transformer_config=transformer_config,
    default_text_column="text_clean",
)
pd.DataFrame([
    {
        "model_name": config["model_name"],
        "model_name_short": config["model_name_short"],
        "aggregation": config["aggregation"],
        "output_schema": config.get("output_schema"),
        "label_map": config["label_map"],
    }
    for config in expanded
])

,model_name,model_name_short,aggregation,output_schema,label_map
0,gtfintechlab/FOMC-RoBERTa,fomc,bysentence,shares,"{'LABEL_0': 1, 'LABEL_1': -1, 'LABEL_2': 0}"
1,ProsusAI/finbert,finbertpro,bysentence,shares,"{'neutral': 0, 'positive': 1, 'negative': -1}"


## 2. Load the real FOMC dataset

This mirrors `autoecon_demo.ipynb`: load the included FOMC statements through `AutoEconSentiment`, then clean the text before transformer scoring.

In [21]:
import_file_path = REPO_ROOT / "data/raw/basic_tests/monetary_policy_statement.parquet.gzip"
export_path = REPO_ROOT / "data/sentiment/transformer_demo_run/"

analyzer = AutoEconSentiment(
    import_file_path=import_file_path,
    text_column="text",
    date_column="date",
    export_path=export_path,
)

df_raw = analyzer.load_data()
df_clean = analyzer.clean_data(clean_config={"tokenize": False, "stem": False})

df_clean[["id_text", "date", "text_clean"]].head(2)

,id_text,date,text_clean
245,1,1994-02-04,Chairman Alan Greenspan announced today that t...
246,2,1994-03-22,Chairman Alan Greenspan announced today that t...


## 3. Prepare real sentence-level rows

Sentence aggregation expects repeated `id_text` values. Each sentence from the cleaned FOMC statements is scored separately, then grouped back to the source document.

In [22]:
sentence_pattern = r"(?<=[.!?])\s+"

actual_sentences = (
    df_clean[["id_text", "date", "text_clean"]]
    .assign(text_clean=lambda df: df["text_clean"].str.split(sentence_pattern, regex=True))
    .explode("text_clean")
    .assign(text_clean=lambda df: df["text_clean"].str.strip())
    .query("text_clean != ''")
    .reset_index(drop=True)
)
actual_sentences["sentence_number"] = actual_sentences.groupby("id_text").cumcount() + 1

actual_sentences.head(10)

,id_text,date,text_clean,sentence_number
0,1,1994-02-04,Chairman Alan Greenspan announced today that t...,1
1,1,1994-02-04,This action is expected to be associated with ...,2
2,2,1994-03-22,Chairman Alan Greenspan announced today that t...,1
3,2,1994-03-22,The action is expected to be associated with a...,2
4,2,1994-03-22,The decision was taken to move toward a less a...,3
5,2,1994-03-22,Chairman Greenspan decided to announce this ac...,4
6,3,1994-04-18,Chairman Alan Greenspan announced today that t...,1
7,3,1994-04-18,This action is expected to be associated with ...,2
8,4,1994-05-17,The Federal Reserve today announced two action...,1
9,4,1994-05-17,The Board approved an increase in the discount...,2


## 4. Run the real FOMC-RoBERTa model

This scores the actual cleaned FOMC sentence rows with `gtfintechlab/FOMC-RoBERTa`, then aggregates sentence classifications back to each source document.

In [23]:
real_transformer = SentimentTransformers(
    df_input=actual_sentences,
    text_column="text_clean",
    model_name="gtfintechlab/FOMC-RoBERTa",
    model_name_short="fomc",
    num_labels=3,
    max_length=512,
    batch_size=16,
    output_schema="shares",
    net_sentiment_formula="positive_minus_negative",
    label_map={"LABEL_0": 1, "LABEL_1": -1, "LABEL_2": 0},
)
real_scores, real_sentence_probabilities = real_transformer.sentiment_pipeline(
    aggregation="bysentence",
    sentence_probability_cutoff=0.7,
)

real_scores.head(10)

INFO:auto_econ_sentiment.models.sentiment_transformers.SentimentTransformers:Loading transformer model: gtfintechlab/FOMC-RoBERTa
INFO:auto_econ_sentiment.models.sentiment_transformers.SentimentTransformers:Transformer model loaded on device: cpu
Transformer Sentiment (fomc): 100%|██████████| 661/661 [03:43<00:00,  2.96it/s]


,fomc_countsentence_LABEL_0,fomc_countsentence_LABEL_1,fomc_countsentence_LABEL_2,fomc_sentiment_bysentence,fomc_count_positive,fomc_count_neutral,fomc_count_negative,fomc_share_positive,fomc_share_neutral,fomc_share_negative,fomc_net_sentiment
id_text,,,,,,,,,,,
1,0,2,0,-1.000000,0,0,2,0.000000,0.000000,1.000000,-1.000000
2,0,4,0,-1.000000,0,0,4,0.000000,0.000000,1.000000,-1.000000
3,0,2,0,-1.000000,0,0,2,0.000000,0.000000,1.000000,-1.000000
4,0,4,4,-0.500000,0,4,4,0.000000,0.500000,0.500000,-0.500000
5,0,2,4,-0.333333,0,4,2,0.000000,0.666667,0.333333,-0.333333
6,0,4,5,-0.444444,0,5,4,0.000000,0.555556,0.444444,-0.444444
7,0,4,4,-0.500000,0,4,4,0.000000,0.500000,0.500000,-0.500000
8,1,2,1,-0.250000,1,1,2,0.250000,0.250000,0.500000,-0.250000
9,7,3,17,0.148148,7,17,3,0.259259,0.629630,0.111111,0.148148


In [24]:
# actual_sentences.to_parquet(
#     export_path / "actual_sentences.parquet.gzip",
#     compression="gzip",
#     index=False,
# )
# real_scores.reset_index().to_parquet(
#     export_path / "sentiment_transformer_fomc_real.parquet.gzip",
#     compression="gzip",
#     index=False,
# )
# real_sentence_probabilities.reset_index().to_parquet(
#     export_path / "sentiment_transformer_fomc_real_sentence_probabilities.parquet.gzip",
#     compression="gzip",
#     index=False,
# )

# {
#     "documents_scored": real_scores.shape[0],
#     "sentences_scored": real_sentence_probabilities.shape[0],
#     "output_path": str(export_path),
# }

In [ ]:
real_scores[
    [
        "fomc_count_positive",
        "fomc_count_neutral",
        "fomc_count_negative",
        "fomc_share_positive",
        "fomc_share_neutral",
        "fomc_share_negative",
        "fomc_net_sentiment",
        "fomc_sentiment_bysentence",
    ]
].head(10)

In [ ]:
real_sentence_probabilities.head(10)